<a href="https://colab.research.google.com/github/nmwaura4/Projects/blob/main/Copy_of_Monty_Data_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,root_mean_squared_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
#

# Exploratory Data Analysis(EDA)

In [ ]:
df=pd.read_excel('/content/Monty_3.xlsx')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df_selected=['Moisture content montdorensis counted','montdorensis count (/g)',
 'Ratio count',
 'carpo eggs/Tm mobiles count',
 'moisture content carpo',
 'carpo count (/g)',
 'carpo eggs count (/g)',
 'montdorensis/g at start up',
 'Ratio at start up',
 'carpo eggs/Tm at start up',
 '% eggs Tm','Average RH room',
 'Average temp (C)',
 'Date_Column']

In [ ]:
columns_to_select = ['Moisture content montdorensis counted','montdorensis count (/g)',
 'Ratio count',
 'carpo eggs/Tm mobiles count',
 'moisture content carpo',
 'carpo count (/g)',
 'carpo eggs count (/g)',
 'montdorensis/g at start up',
 'Ratio at start up',
 'carpo eggs/Tm at start up',
 '% eggs Tm','Average RH room',
 'Average temp (C)',]

df_selected=df[columns_to_select]
df_selected.head()

In [ ]:
df_selected.info()

In [ ]:
df_selected.isnull().sum()

In [ ]:
df_selected.fillna("Average RH room")

In [ ]:
for col in df_selected.columns:
    if df_selected[col].dtype == 'object':
        # Convert to numeric, coercing errors to NaN
        df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce')
        # Fill NaNs with the mode for these (originally object) columns
        # Mode might return multiple values, so take the first one if there are ties
        df_selected[col].fillna(df_selected[col].mode()[0], inplace=True)
    else:
        # Fill NaNs with the mean for numeric columns
        df_selected[col].fillna(df_selected[col].mean(), inplace=True)

# Feature Engeneering

In [ ]:
df_selected.dtypes

In [ ]:
corr = df_selected.corr()
corr

# Correlation Matrix

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
#sample 'Date' column for df, assuming 310 entries matching the DataFrame length.
df['Date'] = pd.date_range(start='2026-01-01', periods=len(df), freq='D')

df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
selected_columns = [
    'Moisture content montdorensis counted',
    'montdorensis count (/g)',
    'Ratio count',
    'carpo eggs/Tm mobiles count',
    'moisture content carpo',
    'carpo count (/g)',
    'carpo eggs count (/g)',
    'montdorensis/g at start up',
    'Ratio at start up',
    'carpo eggs/Tm at start up',
    '% eggs Tm',
    'Average RH room',
    'Average temp (C)',
    'Date'
]

df_selected = df[selected_columns].copy()


for col in df_selected.columns:
    if df_selected[col].dtype == 'object':
        # Convert to numeric, coercing errors to NaN
        df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce')
        # Fill NaNs with the mode for these (originally object) columns
        df_selected[col].fillna(df_selected[col].mode()[0], inplace=True)
    elif pd.api.types.is_datetime64_any_dtype(df_selected[col]):
        # Convert datetime to numerical (e.g., Unix timestamp in seconds)
        df_selected[col] = df_selected[col].astype(np.int64) // 10**9
    else:
        # Fill NaNs with the mean for numeric columns
        df_selected[col].fillna(df_selected[col].mean(), inplace=True)

X = df_selected.drop('montdorensis count (/g)', axis=1)
y = df_selected['montdorensis count (/g)']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model=RandomForestRegressor(n_estimators=100,random_state=42)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
#

In [ ]:
Date=pd.date_range(start='2026-01-01',periods=len(X_test),freq='D')

In [ ]:
pd.DataFrame({'Date':2026,'Actual':y_test,'Predicted':y_pred})

### Real Data Prediction for January to April

In [ ]:
start_date = '2026-01-01'
end_date = '2026-04-30'

# Convert start_date and end_date to Unix timestamps for filtering X
start_timestamp = pd.Timestamp(start_date).timestamp()
end_timestamp = pd.Timestamp(end_date).timestamp()

X_future_real_data = X[(X['Date'] >= start_timestamp) & (X['Date'] <= end_timestamp)].copy()

# Check if any data was found for the period
if X_future_real_data.empty:
    print(f"No historical data found for the period {start_date} to {end_date}.")
else:
    # Make predictions using the actual historical data for the specified date range
    y_future_real_pred = model.predict(X_future_real_data)
    #prediction using real data
    predictions_real_df = pd.DataFrame({
        'Date': pd.to_datetime(X_future_real_data['Date'], unit='s'),
        'Predicted_montdorensis_count': y_future_real_pred
    })

    display(predictions_real_df.head(5))
    display(predictions_real_df.tail(5))

In [ ]:
predictions_real_df.to_csv('predictions_real_data.csv', index=False)
from google.colab import files
files.download('predictions_real_data.csv')

# Importance Feature

In [ ]:
importances = model.feature_importances_
feature_names = X.columns

feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sort the DataFrame by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Display the DataFrame
print(feature_importance_df)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')

#Evaluation Metrics

In [ ]:
print (f"Mean Squared Error: {mean_squared_error(y_test, y_pred):.2f}")
print (f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f}")
print (f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print (f"R-squared: {r2_score(y_test, y_pred):.3f}")

### Actual vs. Predicted Values Plot

In [ ]:
test_results_df = pd.DataFrame({
    'Date': pd.to_datetime(X_test['Date'], unit='s'), # Convert Unix timestamp back to datetime
    'Actual_montdorensis_count': y_test,
    'Predicted_montdorensis_count': y_pred
})

# Sort by Date to ensure proper time series plotting
test_results_df = test_results_df.sort_values(by='Date').reset_index(drop=True)

plt.figure(figsize=(15, 7))
sns.lineplot(x='Date', y='Actual_montdorensis_count', data=test_results_df, label='Actual Count', marker='o')
sns.lineplot(x='Date', y='Predicted_montdorensis_count', data=test_results_df, label='Predicted Count', marker='x')

plt.title('Actual vs. Predicted montdorensis count over Time (Test Set)')
plt.xlabel('Date')
plt.ylabel('montdorensis count (/g)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
test_results_df.to_csv('actual_vs_predicted.csv', index=False)
from google.colab import files
files.download('actual_vs_predicted.csv')

In [ ]:
import joblib

In [ ]:
joblib.dump(model,'model.pkl')